In [ ]:
#pip install ipywidgets

In [ ]:
import pandas as pd
import os
from nilearn import plotting, image, datasets
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import re
import ipywidgets as widgets
from IPython.display import display
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap, BoundaryNorm
from nilearn.image import new_img_like
from scipy.ndimage import label


In [ ]:
# CHOOSE

results_dir = Path("/Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/")
output_path = "sum_overlap_all_contrasts_virtues.nii.gz"
name_title = "virtues"

# results_dir = Path("/Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/vices/")
# output_path = "sum_overlap_all_contrasts_vices.nii.gz"
# name_title = "vices"





In [ ]:
# --- Load all .nii/.nii.gz files of virtues of vices --- (use when not want to separately look at one foundation)
contrast_files = sorted(results_dir.glob("*.nii*"))
print(f"Found {len(contrast_files)} contrast maps.")


In [ ]:

mni152 = datasets.load_mni152_template()
ref_img = mni152
print("Reference MNI152 shape:", ref_img.shape)

sum_data = np.zeros(ref_img.shape, dtype=np.int32)

for f in contrast_files:
    img = image.load_img(f)
    # Resample
    img_resampled = image.resample_to_img(img, ref_img, interpolation='nearest',force_resample=True,copy_header=True)
    data = img_resampled.get_fdata()
    
    # Binarize: positive 1, else 0
    binary_data = (data > 0).astype(np.int32)
    sum_data += binary_data

sum_img = image.new_img_like(ref_img, sum_data)

sum_img.to_filename(output_path)

print(" Saved summed overlap image to:", output_path)

 
sum_data = sum_img.get_fdata()
sum_data[sum_data == 0] = np.nan
sum_img_masked = image.new_img_like(sum_img, sum_data)


In [ ]:

# cmap = plt.cm.YlOrRd

# fig, axes = plt.subplots(
#     3, 1, figsize=(15, 12),
#     gridspec_kw={'hspace': 0.05, 'bottom': 0.15}  # leave space at bottom
# )

# plot_args = dict(
#     threshold=0,
#     cmap=cmap,
#     colorbar=False,
#     draw_cross=False,
#     vmax=7,  # maximum number of overlaps
# )

# plotting.plot_stat_map(sum_img, display_mode='x', cut_coords=5, axes=axes[0], **plot_args)
# plotting.plot_stat_map(sum_img, display_mode='y', cut_coords=5, axes=axes[1], **plot_args)
# plotting.plot_stat_map(sum_img, display_mode='z', cut_coords=5, axes=axes[2], **plot_args)

# sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=len(contrast_files)))
# cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])  # [left, bottom, width, height]
# cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
# cbar.set_label('Number of contrasts', labelpad=2)

# fig.suptitle([name_title], fontsize=14, y=0.95)  # y controls vertical position


# plt.subplots_adjust(top=0.98, bottom=0.12, left=0.02, right=0.98, hspace=0.02)

# fig.savefig(f"1_{name_title}.png", dpi=300, bbox_inches='tight')


# plt.show()


In [ ]:

# fig = plt.figure(figsize=(8, 6))
# display = plotting.plot_glass_brain(
#     sum_img,
#     threshold=0.01,
#     colorbar=False,
#     display_mode='lyrz',  # left, right, sagittal, axial
#     plot_abs=False,
#     cmap=cmap,
#     figure=fig
# )

# cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.03])  # [left, bottom, width, height]
# sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=len(contrast_files)))  # continuous scale
# cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
# cbar.set_label('Number of contrasts', fontsize=10)

# # --- Add title ---
# fig.suptitle([name_title], fontsize=14, y=0.95)  # y controls vertical position

# plt.tight_layout()

# fig.savefig(f"2_{name_title}.png", dpi=300, bbox_inches='tight')

# plt.show()

In [ ]:
# --- Threshold sum image (show voxels with at least 1 contrast active) ---
sum_data = sum_img.get_fdata()
thresholded_sum_data = np.where(sum_data >= 1, sum_data, 0)
sum_img_thresholded = image.new_img_like(sum_img, thresholded_sum_data)

# --- Interactive viewer ---
interactive_view = plotting.view_img(
    sum_img_thresholded,
    threshold=0.1,      # optional: small positive threshold to ignore zeros
    cmap='YlOrRd',      # continuous warm colormap
    black_bg=False,
)

# --- Increase size in notebook ---
interactive_view.width = 1000
interactive_view.height = 800

interactive_view
